# **Recency , Frequency, Monetary (RFM) Customer Segmentation**

## 1\. customer with Sales Recency, Frequency, Monetary

In [25]:
DROP TABLE IF EXISTS #CUST_RAW
SELECT MAIN_IDENTIFIER_NO, LAST_DATE
        ,COALESCE(RECENCY,730) AS RECENCY
        ,COALESCE(FREQUENCY,0) AS FREQUENCY
        ,COALESCE(MONETARY,0) AS MONETARY
INTO #CUST_RAW
FROM sas..CLM_MST_CUSTOMER_CAR_BASE AS CUST
LEFT JOIN (SELECT RWD_CARD_NO
            ,MAX(BILLING_DATE) AS LAST_DATE
            ,DATEDIFF(DAY,DATEADD(Month,-24,'2025-06-30'),MAX(BILLING_DATE)) AS RECENCY
            ,COUNT(DISTINCT BILLING_DATE) AS FREQUENCY
            ,SUM(NET_ITEM_AMT) AS MONETARY
            FROM sas..TRN_BILLING_ITEM_CAR_BASE
            WHERE  BILLING_DATE >= DATEADD(Month,-24,'2025-06-30') AND RWD_CARD_NO IS NOT NULL
            GROUP BY RWD_CARD_NO) AS SALES_2Y
ON CUST.MAIN_IDENTIFIER_NO = RWD_CARD_NO
WHERE MAIN_IDENTIFIER_NO IS NOT NULL AND CUS_STATUS = 'A' AND RWD_CARD_NO IS NOT NULL AND MONETARY > 0

SELECT TOP 5*
FROM #CUST_RAW

(4262463 rows affected)

(5 rows affected)

Total execution time: 00:00:37.540

MAIN_IDENTIFIER_NO,LAST_DATE,RECENCY,FREQUENCY,MONETARY
2006434281,2025-05-24,694,1,2647.1300000000006
1010267591,2025-07-05,736,14,97677.92
1096262689,2025-03-25,634,2,20228.609999999993
1020398132,2025-04-16,656,3,1259
1005900701,2025-06-08,709,15,42070.36


## 2\. RECENCY, FEQUENCY, MONETARY to Decile RFM

In [26]:
DROP TABLE IF EXISTS #CUST_QT
SELECT *
        ,NTILE(10) OVER(ORDER BY RECENCY DESC) AS R 
        ,NTILE(10) OVER(ORDER BY FREQUENCY ASC) AS F
        ,NTILE(10) OVER(ORDER BY MONETARY ASC) AS M
INTO #CUST_QT
FROM #CUST_RAW


SELECT TOP 5*
FROM #CUST_QT

(4262463 rows affected)

(5 rows affected)

Total execution time: 00:00:30.066

MAIN_IDENTIFIER_NO,LAST_DATE,RECENCY,FREQUENCY,MONETARY,R,F,M
1081962075,2024-04-16,291,1,3.694822225952521E-13,9,2,1
2007537523,2025-06-28,729,1,1,1,1,1
2007564772,2025-06-26,727,1,1,1,1,1
2005601479,2025-02-28,609,1,2,5,1,1
2004735038,2025-02-23,604,1,2,5,1,1


## 3.  RFM segment , RFM Score

In [27]:
DROP TABLE IF EXISTS #CUST_RFM
SELECT *
        ,CONCAT(R,F,M) AS RFM_SEGMENT
        ,SUM(R+F+M) AS RFM_SCORE
INTO #CUST_RFM
FROM #CUST_QT
GROUP BY MAIN_IDENTIFIER_NO, LAST_DATE, RECENCY, FREQUENCY, MONETARY, R, F, M
ORDER BY SUM(R+F+M) DESC

SELECT TOP 10*
FROM #CUST_RFM

(4262463 rows affected)

(10 rows affected)

Total execution time: 00:00:01.984

MAIN_IDENTIFIER_NO,LAST_DATE,RECENCY,FREQUENCY,MONETARY,R,F,M,RFM_SEGMENT,RFM_SCORE
1001766920,2025-06-23,724,32,164622.02000000002,1,10,10,11010,21
1001767128,2025-03-23,632,3,34311.89,5,5,8,558,18
1001767268,2024-12-14,533,3,2105,7,5,3,753,15
1001767543,2025-06-26,727,32,92322.84999999999,1,10,10,11010,21
1001767586,2025-05-02,672,8,26052,4,8,8,488,20
1001767616,2024-09-11,439,6,3150.39,8,7,3,873,18
1001767730,2024-06-17,353,13,128960.08,8,9,10,8910,27
1001767772,2024-11-10,499,1,562,7,2,1,721,10
1001767870,2024-06-30,366,3,5615,8,6,5,865,19
1001768043,2025-01-26,576,9,5592,6,8,5,685,19


## 4.Metrics per RFM score

In [28]:
DROP TABLE IF EXISTS #CUST_SCORE
SELECT RFM_SCORE
        ,AVG(RECENCY) AS RECENCY_MEAN
        ,AVG(FREQUENCY) AS FREQUENCY_MEAN
        ,AVG(MONETARY) AS MONETARY_MEAN
        ,COUNT(MONETARY) AS MONETARY_COUNT
INTO #CUST_SCORE
FROM #CUST_RFM
GROUP BY RFM_SCORE
ORDER BY RFM_SCORE

SELECT *
FROM #CUST_SCORE
ORDER BY RFM_SCORE

(28 rows affected)

(28 rows affected)

Total execution time: 00:00:00.109

RFM_SCORE,RECENCY_MEAN,FREQUENCY_MEAN,MONETARY_MEAN,MONETARY_COUNT
3,730,1,352.11600926998835,8630
4,720,1,720.9822410656656,17041
5,706,1,1104.1731199536555,27619
6,687,1,1444.4904346135888,38586
7,664,1,1720.7935757393416,53189
8,637,1,2009.3853112704762,75995
9,638,1,2788.6266855724975,81809
10,611,1,2919.425498031623,119641
11,558,1,2989.393368102934,176687
12,525,1,3555.3593518303474,219603


## 5.Name Segment

In [38]:
SELECT SCORE.RFM_SCORE
        ,CASE WHEN SCORE.RFM_SCORE <= 30 AND SCORE.RFM_SCORE >= 25 THEN 'Platinum'
            WHEN SCORE.RFM_SCORE < 25 AND SCORE.RFM_SCORE >= 19 THEN 'Gold'
            WHEN SCORE.RFM_SCORE < 19 AND SCORE.RFM_SCORE >= 13 THEN 'Silver'
            WHEN SCORE.RFM_SCORE < 13 AND SCORE.RFM_SCORE >= 7 THEN 'Bronze'
            WHEN SCORE.RFM_SCORE < 7 AND SCORE.RFM_SCORE >= 3 THEN 'Churn_Risk'
            END AS SEGMENT_NAME
        ,AVG(RECENCY) AS RECENCY_MEAN
        ,AVG(FREQUENCY) AS FREQUENCY_MEAN
        ,AVG(MONETARY) AS MONETARY_MEAN
        ,COUNT(MONETARY) AS MONETARY_COUNT
FROM #CUST_SCORE AS SCORE
LEFT JOIN (SELECT * FROM #CUST_RFM) AS RFM
ON SCORE.RFM_SCORE = RFM.RFM_SCORE
GROUP BY CASE WHEN SCORE.RFM_SCORE <= 30 AND SCORE.RFM_SCORE >= 25 THEN 'Platinum'
            WHEN SCORE.RFM_SCORE < 25 AND SCORE.RFM_SCORE >= 19 THEN 'Gold'
            WHEN SCORE.RFM_SCORE < 19 AND SCORE.RFM_SCORE >= 13 THEN 'Silver'
            WHEN SCORE.RFM_SCORE < 13 AND SCORE.RFM_SCORE >= 7 THEN 'Bronze'
            WHEN SCORE.RFM_SCORE < 7 AND SCORE.RFM_SCORE >= 3 THEN 'Churn_Risk'
            END
        ,SCORE.RFM_SCORE
ORDER BY SCORE.RFM_SCORE DESC

(28 rows affected)

Total execution time: 00:00:00.406

RFM_SCORE,SEGMENT_NAME,RECENCY_MEAN,FREQUENCY_MEAN,MONETARY_MEAN,MONETARY_COUNT
30,Platinum,144,29,184839.4906666667,150
29,Platinum,214,23,160870.09571552492,1153
28,Platinum,281,19,148488.02491410598,4133
27,Platinum,346,17,130261.07421635647,11102
26,Platinum,401,15,114646.36009774327,25884
25,Platinum,456,15,99968.02545913828,52686
24,Gold,503,14,88598.74208230298,93954
23,Gold,545,16,85281.76793745806,154145
22,Gold,574,17,86779.53997438273,237104
21,Gold,595,23,112080.0416452705,344190
